# Embeddings con sentence-transformers

In [1]:
from sentence_transformers import SentenceTransformer
import torch
import re
import numpy as np
import pandas as pd

In [2]:
# Definimos que use el GPU para esta monda
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando: {device}")

Usando: cuda


In [3]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)
print("Modelo cargado. Dimensión de salida:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado. Dimensión de salida: 384


Cargamos los datasets

In [4]:
conversaciones = pd.read_parquet("../data/dataset_conversaciones/dataset_50k_anonymized.parquet")

In [5]:
conv_full = (conversaciones
    .sort_values(["conv_id", "date"])
    .groupby("conv_id")
    .agg({
        "user_id": "first",
        "input": lambda x: " ".join(x.astype(str)),
        "channel_source": "first"
    })
    .reset_index()
    .rename(columns={"input": "texto_usuario"}))

print(f"Conversaciones reconstruidas: {len(conv_full):,}")
conv_full.head(2)

Conversaciones reconstruidas: 24,119


,conv_id,user_id,texto_usuario,channel_source
0,0000ACA6-AA00-4227-BCE7-5E78C2742D88,USR-12729,Cuál es su tasa de interés para el crédito de ...,1
1,00043647-377f-4528-b2b3-425afd81f6bd,USR-11649,No le deja seleccionar el cobro para hacer la ...,1


In [6]:
def limpiar(t):
    t = str(t).lower()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"http\S+", "", t)
    return t.strip()
    
conv_full["texto_limpio"] = conv_full["texto_usuario"].apply(limpiar)

# Filtrar conversaciones muy cortas (ruido)
conv_full = conv_full[conv_full["texto_limpio"].str.len() > 10].reset_index(drop=True)
print(f"Conversaciones tras filtrar: {len(conv_full):,}")

Conversaciones tras filtrar: 23,791


In [7]:
textos = conv_full["texto_limpio"].tolist()

embeddings = model.encode(
    textos,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Shape embeddings: {embeddings.shape}")
# Esperado: (n_conversaciones, 384)

Batches:   0%|          | 0/372 [00:00<?, ?it/s]

Shape embeddings: (23791, 384)


In [8]:
np.save("embeddings_conv.npy", embeddings)
conv_full[["conv_id", "user_id"]].to_parquet("conv_index.parquet", index=False)
print("Guardado")

Guardado
